# 03 — Activation-Based Attention Heatmaps

This notebook generates activation-based attention heatmaps that visualize what the CBAM-enhanced YOLOv8n model "looks at" when detecting objects in aerial images.


## Install Dependencies

In [5]:
!pip install ultralytics matplotlib pillow -q

from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

candidate_paths = [
    Path('/content/drive/MyDrive/uav-small-object-detector'),
    Path('/content/drive/MyDrive/uav-small-object-detector/UAV_small_obj_detector'),
    Path('/content/drive/MyDrive/UAV_small_obj_detector'),
]

project_root = next((path for path in candidate_paths if path.exists()), None)
if project_root is None:
    raise FileNotFoundError(
        'Could not find the project folder in Google Drive. '
        'Check MyDrive/uav-small-object-detector and MyDrive/UAV_small_obj_detector.'
    )

os.chdir(project_root)
print(f'Working directory: {project_root}')
!pwd

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/UAV_small_obj_detector
/content/drive/MyDrive/UAV_small_obj_detector


## Load Heatmap Utility

In [6]:
import cv2
import numpy as np
import torch

def _unwrap_tensor(output):
    while isinstance(output, (tuple, list)):
        if not output:
            raise ValueError("Target layer returned an empty tuple/list.")
        output = output[0]
    if not isinstance(output, torch.Tensor):
        raise TypeError(f"Expected tensor-like activation, got {type(output).__name__}.")
    return output

def generate_attention_heatmap(model, image_path, target_layer, save_path):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")

    img_resized = cv2.resize(img_bgr, (640, 640))
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

    img_tensor = torch.from_numpy(img_rgb).permute(2, 0, 1).unsqueeze(0).float()
    device = next(model.model.parameters()).device
    img_tensor = img_tensor.to(device)

    activation = {}

    def hook_fn(_, __, output):
        activation["feat"] = _unwrap_tensor(output).detach()

    handle = target_layer.register_forward_hook(hook_fn)
    try:
        model.model.eval()
        with torch.no_grad():
            _ = model.model(img_tensor)
    finally:
        handle.remove()

    feat = activation.get("feat")
    if feat is None:
        raise RuntimeError("Failed to capture activations from the target layer.")

    heatmap = feat[0].mean(dim=0).cpu().numpy()
    heatmap = np.maximum(heatmap, 0)
    if heatmap.max() > 0:
        heatmap /= heatmap.max()

    heatmap = cv2.resize(heatmap, (640, 640))
    heatmap_u8 = np.uint8(255 * heatmap)
    heatmap_color = cv2.applyColorMap(heatmap_u8, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(img_resized, 0.55, heatmap_color, 0.45, 0)

    cv2.imwrite(save_path, overlay)
    return overlay

print("Heatmap utility loaded.")


Heatmap utility loaded.


## Load CBAM-Trained Model

In [7]:
from ultralytics import YOLO

# Auto-detect the latest CBAM checkpoint from Notebook 02.
weight_candidates = sorted(project_root.glob('runs/detect/cbam_visdrone*/weights/best.pt'))
if not weight_candidates:
    raise FileNotFoundError(
        'Could not find any CBAM weights. Expected a path like '
        "runs/detect/cbam_visdrone*/weights/best.pt"
    )

cbam_weight = weight_candidates[-1]
model = YOLO(str(cbam_weight))
print(f"CBAM-trained model loaded from: {cbam_weight}")

# Print layers to identify target
for i, layer in enumerate(model.model.model):
    print(f"Layer {i}: {type(layer).__name__}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
CBAM-trained model loaded from: /content/drive/MyDrive/UAV_small_obj_detector/runs/detect/cbam_visdrone2/weights/best.pt
Layer 0: Conv
Layer 1: Conv
Layer 2: C2f
Layer 3: Conv
Layer 4: C2f
Layer 5: Conv
Layer 6: C2f
Layer 7: Conv
Layer 8: C2f
Layer 9: SPPF
Layer 10: Upsample
Layer 11: Concat
Layer 12: C2f
Layer 13: Upsample
Layer 14: Concat
Layer 15: C2f
Layer 16: Conv
Layer 17: Concat
Layer 18: C2f
Layer 19: Conv
Layer 20: Concat
Layer 21: C2f
Layer 22: Detect


## Restore VisDrone Dataset If Needed

In [8]:
import shutil
from PIL import Image
from ultralytics.utils.downloads import download

dataset_root = Path('/content/VisDrone')

def visdrone2yolo(root, split, source_name):
    source_dir = root / source_name
    images_dir = root / 'images' / split
    labels_dir = root / 'labels' / split
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)

    for img in (source_dir / 'images').glob('*.jpg'):
        shutil.move(str(img), str(images_dir / img.name))

    for ann in (source_dir / 'annotations').glob('*.txt'):
        img_path = images_dir / ann.with_suffix('.jpg').name
        img_w, img_h = Image.open(img_path).size
        dw, dh = 1.0 / img_w, 1.0 / img_h
        lines = []

        with open(ann, encoding='utf-8') as f:
            rows = [x.split(',') for x in f.read().strip().splitlines()]

        for row in rows:
            if row[4] != '0':
                x, y, w, h = map(int, row[:4])
                cls = int(row[5]) - 1
                x_center = (x + w / 2) * dw
                y_center = (y + h / 2) * dh
                w_norm = w * dw
                h_norm = h * dh
                lines.append(f'{cls} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n')

        with open(labels_dir / ann.name, 'w', encoding='utf-8') as out:
            out.writelines(lines)

if not (dataset_root / 'images' / 'val').exists():
    urls = [
        'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip',
        'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip',
        'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-test-dev.zip',
    ]
    download(urls, dir=dataset_root, threads=4)

    for folder, split in {
        'VisDrone2019-DET-train': 'train',
        'VisDrone2019-DET-val': 'val',
        'VisDrone2019-DET-test-dev': 'test',
    }.items():
        visdrone2yolo(dataset_root, split, folder)
        shutil.rmtree(dataset_root / folder)

print(f"Validation images available: {len(os.listdir('/content/VisDrone/images/val'))}")

Unzipping /content/VisDrone/VisDrone2019-DET-test-dev.zip to /content/VisDrone/VisDrone2019-DET-test-dev...: 100% ━━━━━━━━━━━━ 3223/3223 247.7files/s 13.0s
Unzipping /content/VisDrone/VisDrone2019-DET-train.zip to /content/VisDrone/VisDrone2019-DET-train...: 100% ━━━━━━━━━━━━ 12945/12945 1.3Kfiles/s 10.2s
Validation images available: 548


## Select Target Layer for Attention Heatmaps

In [9]:
# Earlier backbone layers are more stable for the activation-based heatmap path.
target_layers_to_try = [6, 4, 9]
target_layer_idx = None

for idx in target_layers_to_try:
    try:
        print(f"Trying layer {idx}: {type(model.model.model[idx]).__name__}")
        target_layer = model.model.model[idx]
        target_layer_idx = idx
        print(f"Selected model.model.model[{idx}] as target layer")
        break
    except Exception as e:
        print(f"Skipping layer {idx}: {e}")

if target_layer_idx is None:
    raise RuntimeError('Could not select a target layer for heatmap generation.')

Trying layer 6: C2f
Selected model.model.model[6] as target layer


## Generate Attention Heatmaps for 10 Validation Images

In [10]:
import random
import shutil
from pathlib import Path

out_dir = Path('results/gradcam_samples')
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

val_images = sorted(os.listdir('/content/VisDrone/images/val'))
samples = random.sample(val_images, min(10, len(val_images)))
print(f"Selected {len(samples)} random validation images.")

success = 0
for i, img_name in enumerate(samples):
    img_path = f'/content/VisDrone/images/val/{img_name}'
    save_path = out_dir / f'gradcam_{i}.png'

    done = False
    for fallback_idx in target_layers_to_try:
        try:
            fallback_layer = model.model.model[fallback_idx]
            generate_attention_heatmap(model, img_path, fallback_layer, str(save_path))
            print(f"[{i}] Generated with layer {fallback_idx}: {save_path}")
            success += 1
            done = True
            break
        except Exception as e:
            print(f"[{i}] Layer {fallback_idx} failed on {img_name}: {e}")

    if not done:
        print(f"[{i}] Completely failed: {img_name}")

print(f"Total generated: {success}")

Selected 10 random validation images.
[0] Generated with layer 6: results/gradcam_samples/gradcam_0.png
[1] Generated with layer 6: results/gradcam_samples/gradcam_1.png
[2] Generated with layer 6: results/gradcam_samples/gradcam_2.png
[3] Generated with layer 6: results/gradcam_samples/gradcam_3.png
[4] Generated with layer 6: results/gradcam_samples/gradcam_4.png
[5] Generated with layer 6: results/gradcam_samples/gradcam_5.png
[6] Generated with layer 6: results/gradcam_samples/gradcam_6.png
[7] Generated with layer 6: results/gradcam_samples/gradcam_7.png
[8] Generated with layer 6: results/gradcam_samples/gradcam_8.png
[9] Generated with layer 6: results/gradcam_samples/gradcam_9.png
Total generated: 10


## Display 2x5 Grid of Heatmaps

In [11]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, ax in enumerate(axes.flatten()):
    heatmap_path = f'results/gradcam_samples/gradcam_{i}.png'
    if os.path.exists(heatmap_path):
        img = mpimg.imread(heatmap_path)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'Sample {i+1}')
    else:
        ax.text(0.5, 0.5, 'Failed', ha='center', va='center',
                transform=ax.transAxes, fontsize=14)
        ax.axis('off')

plt.suptitle('CBAM Attention Heatmaps on VisDrone Aerial Images',
             fontsize=14)
plt.tight_layout()
plt.savefig('results/gradcam_grid.png', dpi=150)
plt.show()

print("Grid saved to results/gradcam_grid.png")

Output hidden; open in https://colab.research.google.com to view.